# Environment Setup

Shared setup notebook for all Databricks ML pipelines.  
Adds the repo root to `sys.path`, detects the runtime environment, and provides Delta I/O helpers.

**Usage:** Every other notebook calls `%run ./_setup` (or `%run ../_setup`) as its first code cell.

In [ ]:
import sys
import os
from pathlib import Path

# ---------------------------------------------------------------------------
# Detect runtime
# ---------------------------------------------------------------------------
ON_DATABRICKS = "spark" in dir() or "DATABRICKS_RUNTIME_VERSION" in os.environ

# ---------------------------------------------------------------------------
# Add repo root to sys.path so shared/ and industry modules are importable
# ---------------------------------------------------------------------------
if ON_DATABRICKS:
    # In Databricks Repos the notebook path looks like
    # /Workspace/Repos/<user>/<repo>/notebooks/_setup
    _nb_dir = os.getcwd()
    _repo_root = str(Path(_nb_dir).parent)
else:
    # Local JupyterLab: notebooks/ is one level below the repo root
    _repo_root = str(Path("__vsc_nbfile__").resolve().parent.parent)
    if not Path(_repo_root).joinpath("shared").is_dir():
        _repo_root = str(Path.cwd().parent)

if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

print(f"ON_DATABRICKS : {ON_DATABRICKS}")
print(f"Repo root     : {_repo_root}")

In [ ]:
# ---------------------------------------------------------------------------
# Industry configuration
# ---------------------------------------------------------------------------
INDUSTRY_CONFIG = {
    "banking": {
        "schema": "banking",
        "table": "credit_data",
        "experiment_path": "/Experiments/ml_models/banking_credit_scoring",
        "registered_model_name": "banking_credit_scoring_model",
    },
    "insurance": {
        "schema": "insurance",
        "table": "claim_data",
        "experiment_path": "/Experiments/ml_models/insurance_fraud_detection",
        "registered_model_name": "insurance_fraud_detection_model",
    },
    "retail": {
        "schema": "retail",
        "table": "customer_data",
        "experiment_path": "/Experiments/ml_models/retail_customer_churn",
        "registered_model_name": "retail_customer_churn_model",
    },
    "mining": {
        "schema": "mining",
        "table": "equipment_data",
        "experiment_path": "/Experiments/ml_models/mining_equipment_failure",
        "registered_model_name": "mining_equipment_failure_model",
    },
}

In [ ]:
import pandas as pd

_CATALOG = "ml_models"
_LOCAL_DATA_DIR = Path(_repo_root) / "_local_data"


def get_delta_table_name(industry: str) -> str:
    """Return the fully-qualified Delta table name for an industry."""
    cfg = INDUSTRY_CONFIG[industry]
    return f"{_CATALOG}.{cfg['schema']}.{cfg['table']}"


def write_to_delta(pandas_df: pd.DataFrame, industry: str) -> None:
    """Write a pandas DataFrame to a Delta table (Databricks) or CSV (local)."""
    table_name = get_delta_table_name(industry)

    if ON_DATABRICKS:
        spark_df = spark.createDataFrame(pandas_df)  # noqa: F821
        spark_df.write.format("delta").mode("overwrite").saveAsTable(table_name)
        print(f"Wrote {len(pandas_df):,} rows to Delta table: {table_name}")
    else:
        _LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
        csv_path = _LOCAL_DATA_DIR / f"{industry}_{INDUSTRY_CONFIG[industry]['table']}.csv"
        pandas_df.to_csv(csv_path, index=False)
        print(f"Wrote {len(pandas_df):,} rows to CSV: {csv_path}")


def read_from_delta(industry: str) -> pd.DataFrame:
    """Read a Delta table (Databricks) or CSV fallback (local) into pandas."""
    table_name = get_delta_table_name(industry)

    if ON_DATABRICKS:
        df = spark.read.table(table_name).toPandas()  # noqa: F821
        print(f"Read {len(df):,} rows from Delta table: {table_name}")
    else:
        csv_path = _LOCAL_DATA_DIR / f"{industry}_{INDUSTRY_CONFIG[industry]['table']}.csv"
        df = pd.read_csv(csv_path)
        print(f"Read {len(df):,} rows from CSV: {csv_path}")

    return df


print("Setup complete. Helpers available: get_delta_table_name, write_to_delta, read_from_delta")